In [6]:
import pandas as pd
import numpy as np

# ── STEP 0: CONFIG ────────────────────────────────────────────────────────
district_population = {
    'Central': 1200000, 'West': 1100000, 'South': 1150000,
    'North': 1000000, 'East': 950000,
    'Whitefield': 900000, 'Yelahanka': 800000
}

crime_severity_map = {
    'murder': 5, 'murder (gain)': 5, 'dacoity': 4,
    'robbery': 3, 'chain snatching': 2,
    'house thefts': 2, 'ordinary thefts': 2,
    'm.v.thefts': 2, 'servant thefts': 2,
    'hbt (day)': 3, 'hbt( night)': 3,
    'rape': 5, 'molestation': 4,
    'dowry_death': 5, 'domestic_cruelty': 3,
    'trafficking': 4,
    'cybercrime_total': 3
}

# ── STEP 1: CYBERCRIME ────────────────────────────────────────────────────
cyber_real = {
    'Central':    {2021: 660,  2022: 989,  2023: 1126, 2024: 1280},
    'West':       {2021: 1148, 2022: 1384, 2023: 1941, 2024: 2050},
    'North':      {2021: 618,  2022: 1300, 2023: 3260, 2024: 3520},
    'South':      {2021: 942,  2022: 1360, 2023: 2094, 2024: 2250},
    'East':       {2021: 602,  2022: 937,  2023: 1994, 2024: 2105},
    'Whitefield': {2021: 869,  2022: 1055, 2023: 2562, 2024: 2720},
    'Yelahanka':  {2021: 832,  2022: 1550, 2023: 1917, 2024: 2010},
}

cyber_detected_2023 = {
    'Central': 69, 'West': 47, 'North': 238,
    'South': 224, 'East': 129, 'Whitefield': 55, 'Yelahanka': 201
}

cyber_rows = []

for district, years in cyber_real.items():
    for year, cases in years.items():

        detection = round(cyber_detected_2023[district] / cases * 100, 2) \
            if year == 2023 and cases > 0 else None

        cyber_rows.append({
            'year': year,
            'district': district,
            'crime_category': 'cybercrime',
            'crime_type': 'cybercrime_total',
            'cases': cases,
            'detection_rate': detection
        })

cyber_df = pd.DataFrame(cyber_rows)

# ── STEP 2: WOMEN SAFETY ─────────────────────────────────────────────────
district_weights = {
    'Central': 0.18, 'West': 0.16, 'South': 0.17,
    'North': 0.14, 'East': 0.13,
    'Whitefield': 0.12, 'Yelahanka': 0.10
}

women_real = {
    'rape': {2021:116, 2022:152, 2023:176, 2024:189},
    'molestation': {2021:573, 2022:731, 2023:1135, 2024:1210},
    'dowry_death': {2021:26, 2022:29, 2023:25, 2024:30},
    'domestic_cruelty': {2021:422, 2022:580, 2023:696, 2024:730},
    'trafficking': {2021:129, 2022:155, 2023:161, 2024:168}
}

women_rows = []

for crime_type, years in women_real.items():
    for year, total in years.items():
        for district, weight in district_weights.items():
            women_rows.append({
                'year': year,
                'district': district,
                'crime_category': 'women_safety',
                'crime_type': crime_type,
                'cases': round(total * weight),
                'detection_rate': None
            })

women_df = pd.DataFrame(women_rows)

# ── STEP 3: PROPERTY + VIOLENT ───────────────────────────────────────────
raw = pd.read_csv('604a3fc2-e44a-45ce-ab25-3348322bd041.csv')
raw_2024 = pd.read_excel('bengaluru_crime_2024_dataset.xlsx')

crime_map = {
    'Murder': 'violent_crime',
    'Murder (Gain)': 'violent_crime',
    'Dacoity': 'violent_crime',
    'Robbery': 'property_crime',
    'Chain Snatching': 'property_crime',
    'HBT (Day)': 'property_crime',
    'HBT( Night)': 'property_crime',
    'House Thefts': 'property_crime',
    'Servant Thefts': 'property_crime',
    'M.V.Thefts': 'property_crime',
    'Ordinary Thefts': 'property_crime'
}

property_rows = []

for _, row in raw.iterrows():
    ctype = row['Type of Crime']

    if ctype in crime_map:
        for year, col in [(2021,'2021 Reported'),(2022,'2022 Reported'),(2023,'2023 Reported')]:
            det_col = col.replace('Reported','Detected')

            detection = round(row[det_col] / row[col] * 100, 1) if row[col] > 0 else 0

            for district, weight in district_weights.items():
                property_rows.append({
                    'year': year,
                    'district': district,
                    'crime_category': crime_map[ctype],
                    'crime_type': ctype.lower(),
                    'cases': round(row[col] * weight),
                    'detection_rate': detection
                })

# 2024
for _, row in raw_2024.iterrows():
    ctype = row['Crime Head']

    if ctype in crime_map:
        detection = round(row['Detected_2024'] / row['Reported_2024'] * 100, 1) \
            if row['Reported_2024'] > 0 else 0

        for district, weight in district_weights.items():
            property_rows.append({
                'year': 2024,
                'district': district,
                'crime_category': crime_map[ctype],
                'crime_type': ctype.lower(),
                'cases': round(row['Reported_2024'] * weight),
                'detection_rate': detection
            })

property_df = pd.DataFrame(property_rows)

# ── STEP 4: COMBINE ──────────────────────────────────────────────────────
df = pd.concat([cyber_df, women_df, property_df], ignore_index=True)

# ── STEP 5: FEATURE ENGINEERING 🚀 ────────────────────────────────────────

# Severity score
df['severity_score'] = df['crime_type'].map(crime_severity_map).fillna(2)

# Severity level
df['severity_level'] = pd.cut(
    df['severity_score'],
    bins=[0,2,4,5],
    labels=['Low','Medium','High']
)

# Population normalization
df['population'] = df['district'].map(district_population)
df['cases_per_100k'] = (df['cases'] / df['population']) * 100000

# Growth rate
df = df.sort_values(['district','crime_type','year'])
df['growth_rate'] = df.groupby(['district','crime_type'])['cases'].pct_change()

# Violent flag
df['is_violent'] = df['crime_category'].apply(
    lambda x: 1 if x in ['violent_crime','women_safety'] else 0
)

# Crime weight (impact * cases)
df['crime_weight'] = df['cases'] * df['severity_score']

# Risk Index (MAIN ML FEATURE 🔥)
df['risk_index'] = (
    df['cases_per_100k'] * 0.4 +
    df['severity_score'] * 0.3 +
    df['growth_rate'].fillna(0) * 100 * 0.3
)

# Clean types
df['year'] = df['year'].astype(int)
df['cases'] = df['cases'].astype(int)

# ── OUTPUT ───────────────────────────────────────────────────────────────
print("Final Shape:", df.shape)
print(df.head())

df.to_csv('crimesense_master_v4.csv', index=False)

Final Shape: (462, 14)
     year district  crime_category        crime_type  cases  detection_rate  \
252  2021  Central  property_crime   chain snatching     30           100.0   
259  2022  Central  property_crime   chain snatching     27            92.1   
266  2023  Central  property_crime   chain snatching     28            74.5   
427  2024  Central  property_crime   chain snatching     29            79.0   
0    2021  Central      cybercrime  cybercrime_total    660             NaN   

     severity_score severity_level  population  cases_per_100k  growth_rate  \
252               2            Low     1200000        2.500000          NaN   
259               2            Low     1200000        2.250000    -0.100000   
266               2            Low     1200000        2.333333     0.037037   
427               2            Low     1200000        2.416667     0.035714   
0                 3         Medium     1200000       55.000000          NaN   

     is_violent  crime_weig

C:\Users\Shefali\AppData\Local\Temp\ipykernel_24104\3429994421.py:149: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([cyber_df, women_df, property_df], ignore_index=True)
